# Install Requirements

In [1]:
! uv pip install docling
# install faiss as memory vectordb
! uv pip install faiss-gpu-cu12
# evaluation:
! uv pip install ragas
! uv pip install -U langchain_community
! uv pip install unstructured
! uv pip install langchain_huggingface
!uv pip install langchain_groq

Using Python 3.12.13 environment at: /usr
Checked 1 package in 497ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 222ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 309ms
Using Python 3.12.13 environment at: /usr
Resolved 48 packages in 602ms
Prepared 1 package in 3ms
Uninstalled 1 package in 64ms
Installed 1 package in 92ms
 - numpy==2.0.2
 + numpy==2.4.5
Using Python 3.12.13 environment at: /usr
Resolved 70 packages in 193ms
Uninstalled 1 package in 56ms
Installed 1 package in 57ms
 - numpy==2.4.5
 + numpy==2.0.2
Using Python 3.12.13 environment at: /usr
Checked 1 package in 356ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 364ms


In [2]:
# check if you have gpu
!nvidia-smi

Sat May 16 14:55:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Challenge Explanation

> In real-world Retrieval-Augmented Generation (RAG) applications, client data is often provided in a variety of document formats such as PDF (like in our example `AtlasOCR paper`), DOCX, and others. A key challenge is effectively parsing and processing these documents so they can be integrated into a reliable RAG pipeline.

>In this challenge, we will explore the fundamental best practices for building an end-to-end RAG pipeline. The workflow will cover document parsing using Docling, text chunking strategies, embedding generation with open-source Hugging Face models, LLM-based augmentation, and finally, pipeline evaluation using Ragas.


<center>

<img src="https://i.ibb.co/HLXVN9wh/Capture-2026-05-08-214656.png" alt="Capture 2026 05 08 214656" border="0" height="600">

</center>

# Document Parsing

<center>
<img src="https://i.ibb.co/Y7Jkt7n7/Fig-Jam-Untitled-3.png" alt="Fig Jam Untitled (3)" border="0">
</center>

> Docling provides a built-in PDF parser and also supports optional OCR and Vision-Language Model (VLM) enhancements to improve document parsing accuracy. In this challenge, we will use the default configuration, which enables OCR by default.
Additionally, <font color="yellow">**GPU acceleration**</font> will be utilized to speed up document processing and improve overall pipeline efficiency.


In [3]:
from docling.document_converter import DocumentConverter

In [4]:
source = "https://arxiv.org/pdf/2604.08070"
converter = DocumentConverter()
docling_document = converter.convert(source=source).document

[INFO] 2026-05-16 14:56:15,254 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-16 14:56:15,271 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-16 14:56:15,561 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-16 14:56:15,571 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-16 14:56:18,042 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-16 14:56:18,043 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-16 14:56:18,048 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-16 14:56:18,049 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [5]:
# export result as markdown
pdf_markdown = docling_document.export_to_markdown()

In [6]:
# see the results
from IPython.display import Markdown, display
display(Markdown(pdf_markdown))

## AtlasOCR: Building the First Open-Source Darija OCR Model with Vision Language Models

Imane Momayiz, Soufiane Ait Elaouad, Abdeljalil Elmajjodi, Haitame Bouanane

https://www.atlasia.ma/

https://github.com/atlasia-ma/

methodology, spanning data curation, model selection, training strategies, and extensive evaluation.

## Our key contributions include:

- Development of AtlasOCR , the first opensource Darija OCR model achieving state-of-theart performance
- A comprehensive data curation strategy combining synthetic data generation via OCRSmith with diverse real-world Darija content
- Detailed methodology for parameter-efficient fine-tuning using QLoRA and Unsloth frameworks
- Extensive ablation studies optimizing key hyperparameters and training configurations
- Creation of AtlasOCRBench , a publicly available benchmark for Darija OCR evaluation
- Thorough evaluation demonstrating superior Darija performance and competitive results on standard Arabic benchmarks

## 2 Related Work

## 2.1 Arabic OCR Systems

Traditional Arabic OCR systems have primarily focused on Modern Standard Arabic (MSA), with limited attention to dialectal variants. Recent advances in deep learning have improved Arabic text recognition capabilities [10], but dialectal Arabic, particularly Darija, remains underexplored due to data scarcity and linguistic complexity.

## 2.2 Vision Language Models for OCR

Vision Language Models have revolutionized document understanding by integrating visual comprehension with linguistic context [1]. These models excel at zero-shot generalization and have shown promising results in multilingual OCR tasks, making them ideal candidates for low-resource language applications.

## Abstract

Darija, the Moroccan Arabic dialect, is rich in visual content yet lacks specialized Optical Character Recognition (OCR) tools. This paper introduces AtlasOCR , the first open-source Darija OCR model built by fine-tuning a 3B parameter Vision Language Model (VLM). We detail our comprehensive approach, from curating a unique Darija-specific dataset leveraging both synthetic generation with our OCRSmith library and carefully sourced real-world data, to implementing efficient fine-tuning strategies. We utilize QLoRA and Unsloth for parameterefficient training of Qwen2.5-VL 3B and present comprehensive ablation studies optimizing key hyperparameters. Our evaluation on the newly curated AtlasOCRBench and the established KITAB-Bench demonstrates state-of-the-art performance, challenging larger models and highlighting AtlasOCR's robustness and generalization capabilities for both Darija and standard Arabic OCR tasks.

## 1 Introduction

Darija, the Moroccan Arabic dialect, represents a vibrant and visually rich linguistic landscape prevalent in social media, informal documents, and handwritten materials. Despite its widespread use and unique characteristics, the absence of dedicated Optical Character Recognition (OCR) tools for Darija has posed significant barriers for developers, researchers, and organizations working with Moroccan content. This gap hinders digital preservation efforts, limits large-scale text analysis capabilities, and impedes accessibility initiatives.

To address this critical need, we introduce AtlasOCR , the first open-source OCR model specifically designed for Darija. AtlasOCR is a 3-billionparameter model developed by fine-tuning a Vision Language Model (VLM) using parameter-efficient techniques. This paper presents our comprehensive

## 3 Background

## 3.1 The Importance of Darija OCR

The development of Darija OCR capabilities offers substantial benefits across multiple domains:

- Digital Preservation : Converting historical documents, manuscripts, and traditional texts into searchable digital formats
- Social Media Analysis : Enabling large-scale analysis of public discourse and sentiment in Moroccan online content
- Accessibility : Making visual content accessible to screen readers and assistive technologies
- Research Applications : Facilitating linguistic studies, cultural heritage research, and corpus development

## 3.2 Vision Language Model Architecture

Vision Language Models integrate three core components to process multimodal inputs (Figure 1):

- Vision Encoder : Converts input images into high-dimensional vector embeddings capturing visual features such as colors, shapes, textures, and spatial arrangements
- Modality Projection Module : Aligns visual features with the language model's representation space, ensuring semantic consistency
- Language Model : Integrates aligned visual embeddings with textual prompts to generate coherent natural language outputs

Figure 1: Vision Language Model Architecture [1]

<!-- image -->

For OCR applications, this architecture enables understanding both visual text layout and linguistic nuances, crucial for accurately recognizing Darija text across diverse fonts, styles, and backgrounds.

## 4 Data Curation

Developing a robust Darija OCR model required creating a large-scale, diverse dataset reflecting realworld variability. Our hybrid approach combined synthetic data generation with carefully curated realworld samples.

## 4.1 Synthetic Data Generation with OCRSmith

Given the high cost and time requirements for annotating quality datasets in under-resourced languages, synthetic data generation provided an efficient scaling solution. We developed OCRSmith [2], an opensource toolkit that simulates realistic text conditions including various fonts, layouts, backgrounds, and distortions.

OCRSmith enables rapid generation of thousands of labeled images complete with bounding boxes and metadata. Figure 2 demonstrates synthetic Darija text examples generated using this toolkit.

Figure 2: Synthetic Darija Text Examples Generated with OCRSmith

<!-- image -->

## 4.2 Real-World Data Collection

While synthetic data provided scale, real-world images ensured authenticity and captured nuances difficult to simulate. We curated diverse Darija text from multiple contexts:

- Scanned Literature : Two key sources provided approximately 700 pages of high-quality Darija text: ￿￿￿￿￿￿￿' '￿￿￿￿￿￿￿ by Mohammed ElMadlaoui El-Mounabhi and ￿￿￿￿￿' ￿￿￿￿￿￿￿ '￿￿￿￿￿￿￿ by Farouk ElMarrakchi. These were pseudo-labeled using Gemini 2.0 Flash.
- Social Media Content : LinkedIn and similar platforms yielded poster-style educational materials converted to images suitable for OCR training.

- Educational Materials : Moroccan study materials, particularly driving license exam preparations, provided challenging samples with varied text quality and layouts.
- Recipe Collections : Moroccan cookbooks offered domain-specific vocabulary and formatting, carefully preprocessed for optimal OCR training.

Figure 3 illustrates examples from each real-world data source.

<!-- formula-not-decoded -->

## (a) Scanned Literature

<!-- image -->

## (b) Social Media Content

<!-- image -->

(c) Educational Materials

Figure 3: Real-World Darija Text Sources

<!-- image -->

## 4.3 Dataset Composition

Our hybrid approach yielded the first large-scale Darija OCR dataset. Table 1 summarizes the dataset composition, with approximately 86% synthetic and 14% real-world content, ensuring both the scale nec- essary for robust training and the authenticity required for real-world applicability.

Table 1: Darija OCR Dataset Overview

| Split      | Samples   | Total Words   |
|------------|-----------|---------------|
| Train      | 26,162    | 9.5M          |
| Validation | 3,930     | 1.2M          |
| Total      | 30,092    | 10.7M         |

## 5 Methodology

## 5.1 Base Model Selection

Selecting an appropriate base model was critical for developing high-performance Darija OCR within computational constraints. We benchmarked several open-source vision-language models on 55 manually curated real-world Darija images:

- Qwen2-VL 2B [3]
- Qwen2.5-VL 3B [4]
- Qari OCR 2B [5]
- ArabicNougat [6]

We prioritized compact architectures (2B-3B parameters) for efficient training and inference. Evaluation results consistently showed Qwen2.5-VL 3B outperforming alternatives across diverse Darija text domains, establishing it as our base model.

## 5.2 Parameter-Efficient Fine-Tuning Strategy

We employed a parameter-efficient approach combining Quantized Low-Rank Adaptation (QLoRA) with Unsloth optimization:

- QLoRA [7]: Enables efficient fine-tuning by quantizing models to 4-bit precision with lowrank adapters, reducing memory requirements by up to 80% while maintaining performance.
- Unsloth [8]: Accelerates LLM fine-tuning with up to 5× speed improvements and 60% memory reduction through optimized GPU kernels and memory management.

This combination enabled effective 3B parameter model fine-tuning on consumer-grade GPUs, ensuring accessibility and resource efficiency.

## 6 Experimental Design

## 6.1 Ablation Studies

We conducted comprehensive ablation studies to optimize AtlasOCR's performance, focusing on key hyperparameters and training configurations.

## 6.1.1 LoRA Hyperparameters

We explored combinations of LoRA rank ( r ), scaling factor ( α ), and dropout rate, measuring impact via minimum evaluation loss (Table 2).

Table 2: LoRA Hyperparameter Ablation Results

|   r |   α |   Dropout |   Eval Loss |
|-----|-----|-----------|-------------|
|  32 |  32 |      0.00 |      0.2442 |
|  32 |  32 |      0.05 |      0.2456 |
|  64 |  64 |      0.05 |      0.2251 |
| 128 | 128 |      0.05 |      0.2132 |

Results indicate that increasing both rank and scaling factor improves performance, suggesting that additional adapter parameters enhance task-specific feature learning. The optimal configuration was r = 128 , α = 128 , dropout=0.05.

## 6.1.2 Quantization Precision Impact

We compared 4-bit and 16-bit precision effects during fine-tuning (Table 3).

Table 3: Quantization Precision Comparison

| Precision   |   Min Eval Loss |
|-------------|-----------------|
| 4-bit       |          0.2132 |
| 16-bit      |          0.2124 |

The minimal performance difference validates 4-bit quantization for our task, offering substantial memory and computational savings without compromising accuracy.

## 6.1.3 Optimization Parameters

We investigated optimal batch size and learning rate combinations with different LoRA configurations (Tables 4 and 5).

Higher learning rates generally improved convergence speed when not causing instability. Increasing LoRA parameters necessitated learning rate recalibration for optimal performance.

Table 4: Optimization Results (LoRA r = α = 16 )

|   Batch Size |   Learning Rate |   Min Eval Loss |
|--------------|-----------------|-----------------|
|           16 |            2e-4 |          0.3161 |
|           16 |            6e-4 |          0.2326 |
|           16 |            2e-3 |          4.0958 |
|           64 |            2e-4 |          0.3479 |
|          128 |            2e-4 |          0.4086 |
|          128 |            8e-4 |          0.2733 |
|          128 |            2e-3 |          0.2350 |
|          512 |            2e-4 |          0.5725 |

Table 5: Optimization Results (LoRA r = α = 128 )

|   Batch Size |   Learning Rate |   Min Eval Loss |
|--------------|-----------------|-----------------|
|           16 |            6e-5 |          0.2652 |
|           16 |            2e-4 |          0.2132 |
|          128 |            2e-4 |          0.2456 |
|          128 |            8e-4 |          0.2165 |
|          128 |            2e-3 |          8.2561 |

## 6.1.4 Vision Layer Fine-Tuning

We evaluated the impact of freezing vision layers during training (Table 6).

Table 6: Vision Layer Fine-Tuning Impact

| Fine-tune Vision Layers   |   Min Eval Loss |
|---------------------------|-----------------|
| Yes                       |          0.2155 |
| No                        |          0.3173 |

Fine-tuning vision layers significantly improved performance, confirming the benefit of adapting these components to Darija's visual characteristics.

## 6.1.5 RSLoRA Analysis

We tested Rank-Stabilized LoRA (RSLoRA) [9], designed to improve scaling behavior with increasing adapter rank (Table 7).

RSLoRA significantly degraded performance in our setting, indicating it may not be suitable for all model architectures or tasks.

## 7 Evaluation Framework

## 7.1 Benchmark Development

We created AtlasOCRBench , a comprehensive evaluation benchmark tailored for Darija, integrating:

Table 7: RSLoRA Impact Assessment

| RSLoRA Enabled   |   Min Eval Loss |
|------------------|-----------------|
| No               |          0.2132 |
| Yes              |          8.2561 |

- Scanned Darija Literature : High-quality printed text representing foundational document types
- OCRSmith Synthetic Data : Controlled samples testing specific OCR challenges

Our benchmark creation pipeline (Figure 4) employed a two-step process:

Figure 4: Benchmark Creation Pipeline

<!-- image -->

1. Pseudo-labeling : Using Gemini 2.0 Flash with carefully engineered prompts prioritizing human readability
2. Human Annotation : Manual review and correction using Argilla for collaborative editing

The final AtlasOCRBench contains 251 samples, including 55 from scanned literature, ensuring comprehensive coverage of realistic Darija OCR challenges.

## 7.2 Evaluation Metrics

We employed standard OCR evaluation metrics:

- Character Error Rate (CER) : Measures character-level editing distance normalized by ground truth length. Particularly suitable for Darija due to spelling variations and lack of standardized orthography.
- Word Error Rate (WER) : Measures wordlevel editing distance normalized by ground truth word count. While useful, can be misleading for Darija where single character differences mark entire words as incorrect.

We prioritize CER as our primary metric given Darija's linguistic characteristics and flexible spelling conventions.

## 7.3 Preprocessing Protocol

Our evaluation ensures fairness through consistent preprocessing:

## 1. Text Normalization :

- Arabic diacritic removal (harakat)
- Line break standardization and whitespace normalization

## 2. Metric Calculation :

- CER: Space-removed character-level comparison
- WER: Space-tokenized word-level comparison

## 8 Results and Analysis

## 8.1 Benchmark Performance

We evaluated AtlasOCR on both KITAB-Bench [10] (a large-scale Arabic OCR benchmark with 8,800+ samples) and our AtlasOCRBench, providing comprehensive assessment across standard Arabic and Darija tasks.

Table 1:Mean CER and WER performance comparison of evaluated OCR models, sorted by CER (lower is better).AtlasOCR (3B)achieves the best performance.

| Model                |   CER← |   WER↓ |
|----------------------|--------|--------|
| AtlasOCR(3B)         |   0.02 |   0.07 |
| Qwen2.5-VL (7B)      |   0.18 |   0.33 |
| Qwen2.5-VL (3B)      |   0.36 |   0.54 |
| AIN (7B)             |   0.53 |   0.57 |
| Qari-OCR-v0.3-VL(2B) |   0.91 |   0.78 |
| Qwen2-VL(2B)         |   2.62 |   2.63 |

Table 2: Performance comparison of OCR models on KITAB-Bench. AtlasOCR leads among models under 3B parameters and performs competitively with larger open-source models, showing only a major difference with AIN-7B.

<!-- image -->

Figure 5: AtlasOCRBench Results (Lower CER indicates better performance)

| Model               | CER↓           | WER            |
|---------------------|----------------|----------------|
| Open-source<3B      | Open-source<3B | Open-source<3B |
| AtlasOCR(3B)        | 1.43           | 1.47           |
| Qaari (2B)          | 1.80           | 1.93           |
| ArabicNougat (427M) | 4.37           | 4.67           |
| Open-source>3B      | Open-source>3B | Open-source>3B |
| AIN (7B)            | 0.20           | 0.28           |
| Gemma3 (12B)        | 1.05           | 1.45           |
| Qwen2.5-VL (7B)     | 1.20           | 1.41           |
| Qwen2-VL (7B)       | 1.48           | 1.55           |
| Closed-source       | Closed-source  | Closed-source  |
| Gemini-2.0-Flash    | 0.13           | 0.32           |
| GPT-40              | 0.31           | 0.55           |

Figure 6: KITAB-Bench Results (Lower CER indicates better performance)

Figure 5 demonstrates AtlasOCR's state-of-the-art performance on Darija OCR, significantly outperforming all open-source alternatives. Figure 6 shows competitive performance on standard Arabic tasks, challenging much larger models including Gemma3 (12B) [11] and Qwen2.5-VL (7B).

## 8.2 Comparative Analysis

AtlasOCR demonstrates superior performance on Darija-specific tasks while maintaining competitive results on standard Arabic benchmarks. The model achieves state-of-the-art results on AtlasOCRBench, significantly outperforming existing open-source alternatives. On KITAB-Bench, AtlasOCR competes effectively with much larger models, highlighting the effectiveness of our parameter-efficient fine-tuning approach.

Key performance insights include:

- Darija Specialization : AtlasOCR achieves the lowest CER on Darija text, validating our specialized training approach
- Parameter Efficiency : Competitive performance with significantly fewer parameters than larger alternatives
- Cross-lingual Transfer : Strong generalization to standard Arabic despite Darija-focused training

## 9 Discussion

## 9.1 Key Findings

Our experimental results reveal several important insights:

- Parameter Efficiency : AtlasOCR achieves superior performance with fewer parameters than competing models, demonstrating the effectiveness of our fine-tuning strategy
- Cross-lingual Generalization : Strong performance on standard Arabic benchmarks indicates robust generalization capabilities
- Data Strategy Validation : Our hybrid synthetic-real approach proved highly effective for low-resource language OCR development

## 9.2 Limitations and Challenges

Despite strong performance, AtlasOCR has certain limitations:

- Diacritic Handling : Limited capability for recognizing and reconstructing Arabic diacritics when present
- Complex Layout Processing : Performance may degrade on highly complex or artistic document structures
- Domain Specificity : Training data bias toward specific Darija domains may affect performance on underrepresented text types

## 10 Conclusion and Future Directions

AtlasOCR represents a significant advancement in Darija OCR, providing the first open-source solution addressing a critical gap for Moroccan content processing. Through careful dataset curation, efficient fine-tuning strategies, and comprehensive evaluation, we demonstrate that high-performance OCR models can be developed for under-resourced languages within reasonable computational constraints.

## 10.1 Future Research Directions

Our ongoing work focuses on:

- Dataset Enhancement : Expanding coverage of handwritten text, diacritized content, and diverse document types
- Model Compression : Developing sub-3B parameter variants for mobile and edge deployment
- Layout Understanding : Enhancing capabilities for complex document structures and mixed content
- Multilingual Extension : Adapting our methodology for other North African Arabic dialects

## 10.2 Broader Impact

AtlasOCR's development demonstrates the feasibility of creating high-quality language technology for under-resourced languages. Our open-source approach and detailed methodology provide a blueprint for similar initiatives, potentially catalyzing development of specialized tools for other dialectal variants and low-resource languages.

## Acknowledgments

We thank the Moroccan Arabic language community for their support in data collection and validation. Special recognition goes to contributors who helped identify and digitize historical Darija texts, making this work possible.

## References

- [1] Hugging Face Team. NanoVLM: Exploring Vision-Language Models. https: //huggingface.co/blog/nanovlm , 2024.

- [2] AtlasIA Team. OCRSmith: Synthetic OCR Data Generation Toolkit. https://github. com/atlasia-ma/OCRSmith , 2024.
- [3] Qwen Team. Qwen2-VL-2B-Instruct. https://huggingface.co/Qwen/ Qwen2-VL-2B-Instruct , 2024.
- [4] Qwen Team. Qwen2.5-VL-3B-Instruct. https://huggingface.co/Qwen/Qwen2. 5-VL-3B-Instruct , 2024.
- [5] NAMAA-Space Team. Qari-OCRv0.3-VL-2B-Instruct. https:// huggingface.co/NAMAA-Space/Qari-OCR-v0. 3-VL-2B-Instruct , 2024.
- [6] Mohamed Rashad. ArabicNougat: Arabic Document Understanding Model. https://huggingface.co/MohamedRashad/ arabic-large-nougat , 2024.
- [7] Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, and Luke Zettlemoyer. QLoRA: Efficient Finetuning of Quantized LLMs. arXiv preprint arXiv:2305.14314 , 2023.
- [8] Unsloth Team. Unsloth: 5x Faster LLM Finetuning. https://unsloth.ai , 2024.
- [9] Damjan Kalajdzievski et al. RSLoRA: RankStabilized LoRA for Fine-tuning Large Language Models. arXiv preprint , 2024.
- [10] Ahmed Attia et al. KITAB-Bench: A Comprehensive Benchmark for Arabic OCR and Document Understanding. arXiv preprint arXiv:2402.14949 , 2024.
- [11] Gemma Team. Gemma: Open Models Based on Gemini Research and Technology. arXiv preprint arXiv:2403.08295 , 2024.

In [7]:
# save result in file
with open("document.md","w") as md:
  md.write(pdf_markdown)

# Document Chunking

<center>
<img src="https://i.ibb.co/rKRHWFD2/Fig-Jam-Untitled-4.png" alt="Fig Jam Untitled (4)" border="0">
</center>

In [8]:
# document chunking using hybrid method
# 1stp: hybrid method need tokenizer
from transformers import AutoTokenizer
TOKENIZER_MODEL_ID = "sentence-transformers/all-MiniLM-L12-v2"
tokenizer= AutoTokenizer.from_pretrained(TOKENIZER_MODEL_ID)
# test tokenizer:
ids = tokenizer.encode("Hello from abdeljalil")
print(ids)
tokens = tokenizer.convert_ids_to_tokens(ids,skip_special_tokens=True)
print(tokens)

[101, 7592, 2013, 19935, 2884, 28948, 2140, 102]
['hello', 'from', 'abd', '##el', '##jali', '##l']


In [9]:
from docling.chunking import HybridChunker
# 2stp: define max number of token each chunk should contain
MAX_TOKENS = 64
# 3stp: define chunker
chunker = HybridChunker(
    tokenizer = tokenizer,
    max_tokens = MAX_TOKENS,
    merge_peers=True
)

In [10]:
# last step: apply cunker on our document and see the results
chunks = list(chunker.chunk(docling_document))
chunks = [chunk.text for chunk in chunks]
print(f"Number of chunks is: {len(chunks)}")
for i,chunk in enumerate(chunks[:4]):
  print(f"------- chunk_{i+1} -------")
  print(chunk)

Number of chunks is: 110
------- chunk_1 -------
Imane Momayiz, Soufiane Ait Elaouad, Abdeljalil Elmajjodi, Haitame Bouanane
https://www.atlasia.ma/
------- chunk_2 -------
https://github.com/atlasia-ma/
methodology, spanning data curation, model selection, training strategies, and extensive evaluation.
------- chunk_3 -------
- Development of AtlasOCR , the first opensource Darija OCR model achieving state-of-theart performance
- A comprehensive data curation strategy combining synthetic data generation via OCRSmith with diverse real-world Darija content
------- chunk_4 -------
- Detailed methodology for parameter-efficient fine-tuning using QLoRA and Unsloth frameworks
- Extensive ablation studies optimizing key hyperparameters and training configurations
- Creation of AtlasOCRBench , a publicly available benchmark for Darija OCR evaluation


# Chunker Embedding

<center>
<img src="https://i.ibb.co/Z6gbx2Y8/Untitled-Fig-Jam-2.png" alt="Untitled Fig Jam (2)" border="0">
</center>

In [11]:
# load embedding model using sentence transformer
from sentence_transformers import SentenceTransformer
EMBEDDING_MODEL_ID = TOKENIZER_MODEL_ID
embedding_model = SentenceTransformer(EMBEDDING_MODEL_ID)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
# test the model
queries = ["Hello","Hi"]
embeddings = embedding_model.encode(queries)
print(f"embedding result dim: {embeddings.shape}") #dim=> 2: number of queries, 384: embed dim
# calculate similarity
similarity = embedding_model.similarity(embeddings[0],embeddings[1]).squeeze().item()
print(f"similarity result is: {similarity}")

embedding result dim: (2, 384)
similarity result is: 0.8213102221488953


In [13]:
# embed all chunks
chunks_embedding = embedding_model.encode(chunks)
print(f"chunks embedding result dim: {chunks_embedding.shape}")

chunks embedding result dim: (110, 384)


# Store in VectDB/ Index

<center>
<img src="https://i.ibb.co/mCwkdXVP/Untitled-Fig-Jam-3.png" alt="Untitled Fig Jam (3)" border="0">
</center>

> In this challenge, we will use an in-memory vector index, specifically <font color="yellow">**FAISS**</font> by Meta, instead of a dedicated vector database.
>
>In real-world production environments, you may choose to use a vector database such as PGVector, Qdrant, or other scalable vector storage solutions depending on your application requirements.


In [14]:
import faiss
# 1.define the embedding dim
dim = chunks_embedding.shape[-1]
# 2.create index with L2 Euclidean distance
index = faiss.IndexFlatL2(dim)
# 3. add embedding vectors to the index
index.add(chunks_embedding)

In [15]:
# 4. test search
query = ["what is atlasOCR"] # user_queries
# embed query
query_embedding = embedding_model.encode(query)
top_k = 4 # retrieve top_k
distance_scores, indices = index.search(query_embedding,top_k)
print(f"top_k_indices: {indices[0]}")
print(f"similarity_score: {distance_scores[0]}")

top_k_indices: [77 91 93 84]
similarity_score: [0.68363065 0.7599833  0.7637874  0.8325908 ]


In [16]:
# indices to chunks
import numpy as np
chunks_array = np.array(chunks) # make it numpy for easy indixing
retrieved_chunks = chunks_array[indices[-1]].tolist()
for i,(rc,dist) in enumerate(zip(retrieved_chunks,distance_scores[0])):
  print(f"------- chunk_{i+1} -----------")
  print(f"Similarity score: {dist}")
  print(f"Chunk: {rc}")

------- chunk_1 -----------
Similarity score: 0.683630645275116
Chunk: Figure 5: AtlasOCRBench Results (Lower CER indicates better performance)
Open-source<3B, CER↓ = Open-source<3B. Open-source<3B, WER = Open-source<3B. AtlasOCR(3B), CER↓ =
------- chunk_2 -----------
Similarity score: 0.7599833011627197
Chunk: Despite strong performance, AtlasOCR has certain limitations:
- Diacritic Handling : Limited capability for recognizing and reconstructing Arabic diacritics when present
- Complex Layout Processing : Performance may degrade on highly complex or artistic document structures
------- chunk_3 -----------
Similarity score: 0.7637873888015747
Chunk: AtlasOCR represents a significant advancement in Darija OCR, providing the first open-source solution addressing a critical gap for Moroccan content processing.
------- chunk_4 -----------
Similarity score: 0.8325908184051514
Chunk: Figure 5 demonstrates AtlasOCR's state-of-the-art performance on Darija OCR, significantly outperforming al

# Reranker

<center>
<img src="https://i.ibb.co/jd2BW85/Fig-Jam-Untitled-6.png" alt="Fig Jam Untitled (6)" border="0">
</center>

* Reranker model example: https://huggingface.co/jinaai/jina-reranker-v3

In [17]:
# load reranker model from huggingface
from transformers import AutoModel
RERANKER_MODEL_ID = 'jinaai/jina-reranker-v3'
reranker_model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    device_map = "auto", # auto: cuda or cpu or mps
    trust_remote_code=True,
).eval()

Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

In [18]:
# Reranker Query with retrieved chunks and select top_n=2
reranker_result = reranker_model.rerank(query[0],retrieved_chunks, top_n=2)
for result in reranker_result:
  print("--------")
  print(f"Score: {result['relevance_score']}")
  print(f"Document: {result['document']}")

--------
Score: 0.44151052832603455
Document: AtlasOCR represents a significant advancement in Darija OCR, providing the first open-source solution addressing a critical gap for Moroccan content processing.
--------
Score: 0.2015235275030136
Document: Figure 5 demonstrates AtlasOCR's state-of-the-art performance on Darija OCR, significantly outperforming all open-source alternatives.


# LLM Augmentation

<center>
<img src="https://i.ibb.co/k2dkcxbK/Fig-Jam-Untitled-7.png" alt="Fig Jam Untitled (7)" border="0">
</center>

* Groq: https://console.groq.com/keys

In [19]:
# We will use groq models with openai sdk, try to get your api key from groq link above
from openai import OpenAI
import os
BASE_URL = "https://api.groq.com/openai/v1"
API_KEY = os.environ["openai_key"] # add your api key
MODEL = "openai/gpt-oss-120b" # you can change it to any available model from groq
client = OpenAI(
    base_url = BASE_URL,
    api_key = API_KEY
)

In [20]:
# extract content of chunks in reranker
context = [doc["document"] for doc in reranker_result]
context

['AtlasOCR represents a significant advancement in Darija OCR, providing the first open-source solution addressing a critical gap for Moroccan content processing.',
 "Figure 5 demonstrates AtlasOCR's state-of-the-art performance on Darija OCR, significantly outperforming all open-source alternatives."]

In [21]:
# prepare Prompt Template
user_query = input("ask something about AtlasOCR:")
prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{user_query}

If the answer is not in the context, say you do not know.
"""

ask something about AtlasOCR:what is OCR?


In [22]:
# get response
response = client.chat.completions.create(
    model = MODEL,
    messages = [
        {"role":"user","content":prompt}
    ]
)
response.choices[0].message.content

'I do not know.'

In [23]:
def apply_chat_template(context,query):
  prompt = f"""
  Answer the question using ONLY the context below.

  Context:
  {context}

  Question:
  {query}

  If the answer is not in the context, say you do not know.
  """
  return prompt
def chat_document(question,index_top_k=4,reranker_top_n=2):
  # 1. query embedding
  query_embedding = embedding_model.encode([question])
  # 2. index search
  sim,indices = index.search(query_embedding,k=index_top_k)
  # 3. indices to chunks
  selected_chunks = chunks_array[indices].tolist()[0]
  # 4. reranker
  results = reranker_model.rerank(question,selected_chunks,top_n=reranker_top_n)
  context = [r["document"] for r in results]
  # 5. apply chat template
  prompt = apply_chat_template(context,question)
  # 6. generate answer
  response = client.chat.completions.create(
    model = MODEL,
    messages = [
        {"role":"user","content":prompt}
    ]
  )
  return response.choices[0].message.content, context

In [24]:
chat_document("what is atlasocr")

('AtlasOCR is an open‑source optical‑character‑recognition system specifically designed for Darija (Moroccan Arabic). It represents a major advancement in Darija OCR, being the first open‑source tool that fills a critical gap in processing Moroccan content and delivering state‑of‑the‑art performance that surpasses other open‑source alternatives.',
 ['AtlasOCR represents a significant advancement in Darija OCR, providing the first open-source solution addressing a critical gap for Moroccan content processing.',
  "Figure 5 demonstrates AtlasOCR's state-of-the-art performance on Darija OCR, significantly outperforming all open-source alternatives."])

# RAG Pipeline Evaluation

> After building the pipeline, one of the most important steps is evaluating the application to assess aspects such as hallucination, factual accuracy, and overall response quality.
>
> For this purpose, we will use <font color="yellow">**Ragas**</font>, one of the most widely used frameworks for RAG evaluation. With it, we will:
>
> 1. Generate a test dataset.
> 2. Evaluate the performance of the RAG pipeline.


<center>
<img src="https://i.ibb.co/q3vw16z1/Fig-Jam-Untitled-1.png" alt="Fig Jam Untitled (1)" border="0">
</center>

In [25]:
# 1. Load Markdowns
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader
docs = DirectoryLoader("./", glob="*.md").load()

In [26]:
# 2. Load Embedding Model
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
ragas_embedding_model = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_ID)
)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_9351/2540355116.py:4: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embedding_model = LangchainEmbeddingsWrapper(


In [27]:
# 3. Load LLM for generation of testset
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
ragas_generator_llm = LangchainLLMWrapper(
    ChatGroq(
        model=MODEL,
        api_key=API_KEY
    )
)

/tmp/ipykernel_9351/2672612429.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_generator_llm = LangchainLLMWrapper(


In [28]:
# 4. create ragas testset generator
from ragas.testset import TestsetGenerator
generator = TestsetGenerator(
    llm=ragas_generator_llm,
    embedding_model=ragas_embedding_model
)

In [29]:
# 5. generate testset dataset of size 3
dataset = generator.generate_with_langchain_docs(docs, testset_size=3)

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/4 [00:00<?, ?it/s]

In [35]:
# show dataset as pandas
df = dataset.to_pandas()
df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,How does the Qwen2.5-VL model contribute to th...,[AtlasOCR: Building the First Open-Source Dari...,"The Qwen2.5-VL model, a 3 B parameter Vision L...",Darija OCR Researcher,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,what kitab-bench do in darija ocr research?,[2 Related Work\n\n2.1 Arabic OCR Systems\n\nT...,KITAB-Bench is an established benchmark used t...,Darija OCR Researcher,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,How does the character error rate (CER) of Qwe...,[<1-hop>\n\n8 Results and Analysis 8.1 Benchma...,"On the KITAB‑Bench dataset, AtlasOCR (3B) achi...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer
3,How does OCRSmtih contrbute to the creaton of ...,[<1-hop>\n\n4 Data Curation\n\nDeveloping a ro...,OCRSmith is an open‑source synthetic data gene...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


<center>
<img src="https://i.ibb.co/mCBYMcBC/Fig-Jam-Untitled-2.png" alt="Fig Jam Untitled (2)" border="0" height="500">
</center>

In [36]:
from tqdm import tqdm
data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}
for i,row in tqdm(df.iterrows()):
  data["question"].append(row["user_input"])
  answer, context = chat_document(row["user_input"])
  data["answer"].append(answer)
  data["contexts"].append(context)
  data["ground_truth"].append(row["reference"])

4it [00:03,  1.00it/s]


In [37]:
import pandas as pd
pd.DataFrame(data)

,question,answer,contexts,ground_truth
0,How does the Qwen2.5-VL model contribute to th...,I do not know.,"[To address this critical need, we introduce A...","The Qwen2.5-VL model, a 3 B parameter Vision L..."
1,what kitab-bench do in darija ocr research?,KITAB‑Bench is an established benchmark used t...,[Our evaluation on the newly curated AtlasOCRB...,KITAB-Bench is an established benchmark used t...
2,How does the character error rate (CER) of Qwe...,I do not know.,[Table 1:Mean CER and WER performance comparis...,"On the KITAB‑Bench dataset, AtlasOCR (3B) achi..."
3,How does OCRSmtih contrbute to the creaton of ...,OCRSmith is used to generate synthetic Darija ...,"[- Development of AtlasOCR , the first opensou...",OCRSmith is an open‑source synthetic data gene...


<center>
<img src="https://i.ibb.co/nsP74LKw/Untitled-Fig-Jam-5.png" alt="Untitled Fig Jam (5)" border="0">
</center>

In [33]:
from ragas import evaluate
from datasets import Dataset
evaluation_result = evaluate(
    Dataset.from_dict(data),
    llm = ragas_generator_llm,
    embeddings=ragas_embedding_model
)

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[12]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[0]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[1]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()


In [38]:
evaluation_result

{'answer_relevancy': 0.3265, 'context_precision': 0.6667, 'faithfulness': 0.3214, 'context_recall': 0.3333}